# 14 - Qwen3-32B QLoRA Fine-tuning

Heavy same-LLM fine-tuning run. The base checkpoint is `Qwen/Qwen3-32B`; the fine-tuned system will use the same checkpoint plus this LoRA adapter. Benchmark samples remain excluded from training.

In [ ]:
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes peft datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
from src.finetune_lora import train_lora

model_name = 'Qwen/Qwen3-32B'
train_jsonl = DRIVE_ROOT / 'data/processed/finetune_train_combined.jsonl'
val_jsonl = DRIVE_ROOT / 'data/processed/finetune_val_combined.jsonl'
output_dir = DRIVE_ROOT / 'models/adapters/qwen3_32b_qlora_combined_v1'

for path in [train_jsonl, val_jsonl]:
    if not path.exists():
        raise FileNotFoundError(path)

train_jsonl, val_jsonl, output_dir

In [ ]:
# Strong run. If Colab OOMs, reduce max_length to 1536 or lora_r to 16.
# If runtime is too long, set max_train_samples=8000 and max_val_samples=800.
run_config = train_lora(
    train_jsonl=train_jsonl,
    val_jsonl=val_jsonl,
    output_dir=output_dir,
    model_name=model_name,
    max_length=2048,
    max_train_samples=None,
    max_val_samples=None,
    num_train_epochs=1.0,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    logging_steps=25,
    eval_steps=250,
    save_steps=250,
    warmup_ratio=0.03,
    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    use_4bit=True,
)
run_config